# ViHSD Mixture of Experts experiment

This notebook prepares a Colab runtime, but training and evaluation are run manually from shell commands. That keeps the workflow explicit: edit a command, run it, and reuse the resulting `run_id` when evaluating.

Use the command cells below to:
- mount Drive and install the project
- authenticate Hugging Face and W&B through Colab's built-in Secrets panel
- run a smoke test or a full experiment
- evaluate the saved checkpoint
- compare saved metrics in the final summary cell

Create the `HF_TOKEN` and `WANDB_API_KEY` secrets in Colab before running the authentication cell. The credentials are stored by the respective login tools and are not written to the notebook, an `.env` file, or the repository.

The repository code remains the source of truth. Each run saves its resolved configuration, checkpoint, metrics, and predictions.

## 1. Mount Google Drive

The YAML checkpoint path points to `/content/drive/MyDrive/ViHSD-MoE/checkpoints`. Drive must be mounted before training so `.safetensors` files persist after the Colab runtime ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the GitHub repository and install dependencies

This notebook treats GitHub as the source of truth. Each runtime clones the latest `main` branch into `/content/moe-vihsd`, installs dependencies from that clone, and runs the scripts there.

In [ ]:
PROJECT_DIR = '/content/moe-vihsd'
REPOSITORY_URL = 'https://github.com/lngphgthao/moe-vihsd.git'

!rm -rf $PROJECT_DIR
!git clone --depth 1 --branch main $REPOSITORY_URL $PROJECT_DIR
%cd $PROJECT_DIR
%pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Create a Colab Secret named HF_TOKEN before continuing.')
os.environ['HF_TOKEN'] = hf_token

wandb_api_key = userdata.get('WANDB_API_KEY')
if not wandb_api_key:
    raise RuntimeError('Create a Colab Secret named WANDB_API_KEY before continuing.')
os.environ['WANDB_API_KEY'] = wandb_api_key

import wandb
wandb.login(verify=True)
os.environ['CHECKPOINT_DIR'] = '/content/drive/MyDrive/ViHSD-MoE/checkpoints'
os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/ViHSD-MoE/results'
print('Hugging Face and W&B authentication configured from Colab Secrets.')

## 3. Run training and evaluation manually

The repository supports two comparison architectures: `dense_phobert` and `phobert_moe`. Run the smoke test first, then use separate run IDs for the final experiments. Every run saves its resolved configuration, architecture-specific checkpoint, metrics, and predictions.

### Smoke test

```bash
python train.py --config configs/vihsd.yaml --smoke-test --run-id smoke-check
python evaluate.py --config configs/vihsd.yaml --run-id smoke-check
```

### Dense PhoBERT baseline

```bash
python train.py \
  --config configs/vihsd.yaml \
  --no-smoke-test \
  --run-id dense-phobert-integrated \
  --set model.architecture=dense_phobert \
  --set model.pooling=mean \
  --set training.loss_type=cross_entropy
python evaluate.py --config configs/vihsd.yaml --run-id dense-phobert-integrated
```

### PhoBERT MoE

```bash
python train.py \
  --config configs/vihsd.yaml \
  --no-smoke-test \
  --run-id phobert-moe-integrated \
  --set model.architecture=phobert_moe \
  --set training.loss_type=cross_entropy
python evaluate.py --config configs/vihsd.yaml --run-id phobert-moe-integrated
```

Use a unique `--run-id` for every experiment. Inspect `checkpoints/<run-id>/resolved_config.yaml` first when debugging.

## 4. Training command

Edit the command below and run the cell manually. The command prints the run ID and writes the checkpoint and metrics for that run.

For repeated experiments, change `--run-id` and any `--set` values before running the cell. Do not reuse a run ID unless you intentionally want to replace or inspect that run.

In [ ]:
# Edit the run_id before executing this cell.
# Dense PhoBERT baseline:
!python train.py --config configs/vihsd.yaml --no-smoke-test --run-id dense-phobert-integrated --set model.architecture=dense_phobert --set model.pooling=mean --set training.loss_type=cross_entropy

# PhoBERT MoE:
# !python train.py --config configs/vihsd.yaml --no-smoke-test --run-id phobert-moe-integrated --set model.architecture=phobert_moe --set training.loss_type=cross_entropy

## 5. Evaluation command

Run evaluation manually after the training command finishes. Use the same `run_id` so evaluation loads that run's `resolved_config.yaml` and best checkpoint.

In [ ]:
# Use the run_id from the training command above.
!python evaluate.py --config configs/vihsd.yaml --run-id baseline-current-moe

# Or evaluate a checkpoint directly:
# !python evaluate.py --config configs/vihsd.yaml --checkpoint checkpoints/baseline-current-moe/vihsd_moe_best.safetensors

## 6. Command-line argument reference

Run commands from the repository root (`/content/moe-vihsd` in Colab).

### `train.py`

- `--config PATH`: configuration file, default `configs/vihsd.yaml`.
- `--set SECTION.KEY=VALUE`: override an existing YAML value; repeat as needed.
- `--smoke-test` / `--no-smoke-test`: select the short or full training profile.
- `--run-id ID`: unique experiment folder and name.

### Supported architectures

| Architecture | Purpose |
| --- | --- |
| `dense_phobert` | Dense PhoBERT baseline with mean pooling. |
| `phobert_moe` | PhoBERT with token-level MoE layers. |

For dense PhoBERT, always pass `--set model.pooling=mean`. Evaluation uses the same `--run-id` and automatically finds the architecture-specific checkpoint. Existing legacy checkpoints are still supported.